# Silver Layer: Data Integration and Null Handling

This notebook performs data integration (joining Retention and BOB datasets), handles null values according to the specified logic, and saves the integrated dataset to the silver layer.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Set current date
CURRENT_DATE = pd.to_datetime('2026-04-13')

print("Loading cleaned datasets from bronze layer...")

Loading cleaned datasets from bronze layer...


In [6]:
# Load cleaned datasets
date_cols_retention = ['customer_since', 'agreement_end_date', 'case_creation_date', 'resolved_date', 'registered_date', 'expected_pull_date']
date_cols_bob = ['agreement_start_date', 'agreement_end_date']

retention_df = pd.read_csv('../../data/01_raw/cleaned_retention.csv', parse_dates=date_cols_retention)
bob_df = pd.read_csv('../../data/01_raw/cleaned_bob.csv', parse_dates=date_cols_bob)

print(f"Retention shape: {retention_df.shape}")
print(f"BOB shape: {bob_df.shape}")

# Drop rows with null join keys
retention_df = retention_df.dropna(subset=['customer_account_number'])
bob_df = bob_df.dropna(subset=['account_number'])

print(f"After dropping null keys - Retention shape: {retention_df.shape}")
print(f"After dropping null keys - BOB shape: {bob_df.shape}")

Retention shape: (85411, 29)
BOB shape: (376803, 26)
After dropping null keys - Retention shape: (77949, 29)
After dropping null keys - BOB shape: (376784, 26)


In [ ]:
def fill_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col in ['account_number', 'customer_account_number', 'agreement_number']:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            mean_value = df[col].mean(skipna=True)
            if pd.notna(mean_value):
                df[col] = df[col].fillna(mean_value)
        else:
            mode_value = df[col].mode(dropna=True)
            if len(mode_value) > 0:
                df[col] = df[col].fillna(mode_value.iloc[0])
            else:
                df[col] = df[col].fillna('Unknown')
    return df


def fill_missing_values_after_join(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col in ['account_number', 'customer_account_number', 'agreement_number']:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            mean_value = df[col].mean(skipna=True)
            if pd.notna(mean_value):
                df[col] = df[col].fillna(mean_value)
        else:
            mode_value = df[col].mode(dropna=True)
            if len(mode_value) > 0:
                df[col] = df[col].fillna(mode_value.iloc[0])
            else:
                df[col] = df[col].fillna('Unknown')
    return df


def clean_dataset(df: pd.DataFrame, join_key: str, unwanted_columns: list) -> pd.DataFrame:
    df = df.copy()
    df = df.drop(columns=[col for col in unwanted_columns if col in df.columns], errors='ignore')
    df = fill_missing_values(df)
    df = df.dropna(subset=[join_key])
    df = df.drop_duplicates()
    return df


def label_churn(row):
    active = row['active_agreements']
    lost = row['lost_agreements']
    if active == 0:
        return 2
    elif lost == 0:
        return 0
    return 1


def join_datasets(retention_df, bob_df):
    retention_unwanted = [
        'case_id', 'country', 'customer_name', 'customer_since',
        'registered_time', 'resolved_time'
    ]
    bob_unwanted = [
        'postal_code', 'vat_number', 'system_status', 'is_bob', 'bpg',
        'msdyn_product_number', 'product_name', 'machine', 'machine_variant',
        'chemistry'
    ]

    retention_clean = clean_dataset(retention_df, 'customer_account_number', retention_unwanted)
    bob_clean = clean_dataset(bob_df, 'account_number', bob_unwanted)

    joined_df = pd.merge(
        retention_clean,
        bob_clean,
        left_on='customer_account_number',
        right_on='account_number',
        how='left',
        suffixes=('_retention', '_bob')
    )

    # Preserve the original retention account key for unmatched rows
    joined_df['account_number'] = joined_df['account_number'].fillna(joined_df['customer_account_number'])

    for col in joined_df.columns:
        if col.endswith('_bob'):
            base = col[:-4]
            dual = f"{base}_retention"
            if dual in joined_df.columns:
                joined_df = joined_df.drop(columns=[dual])
                joined_df = joined_df.rename(columns={col: base})
    for col in joined_df.columns:
        if col.endswith('_retention'):
            base = col[:-10]
            if base not in joined_df.columns:
                joined_df = joined_df.rename(columns={col: base})

    joined_df = joined_df.dropna(subset=['customer_account_number'])

    agg = joined_df.groupby('account_number', dropna=False).agg(
        total_agreements=('agreement_number', 'nunique')
    ).reset_index()

    lost = joined_df[joined_df['resolution_status'] == 'Customer Lost']
    lost_counts = lost.groupby('account_number', dropna=False)['agreement_number'].nunique().reset_index()
    lost_counts = lost_counts.rename(columns={'agreement_number': 'lost_agreements'})
    agg = agg.merge(lost_counts, on='account_number', how='left')
    agg['lost_agreements'] = agg['lost_agreements'].fillna(0).astype(int)

    active = joined_df[joined_df['resolution_status'] != 'Customer Lost']
    active_counts = active.groupby('account_number', dropna=False)['agreement_number'].nunique().reset_index()
    active_counts = active_counts.rename(columns={'agreement_number': 'active_agreements'})
    agg = agg.merge(active_counts, on='account_number', how='left')
    agg['active_agreements'] = agg['active_agreements'].fillna(0).astype(int)

    agg['churn_category'] = agg.apply(label_churn, axis=1)
    joined_df = joined_df.merge(agg, on='account_number', how='left')

    # Drop duplicate account number column
    joined_df = joined_df.drop(columns=['customer_account_number'], errors='ignore')

    # Fill missing values introduced by the left join
    joined_df = fill_missing_values_after_join(joined_df)
    return joined_df


# Join and clean datasets using embedded preprocessing logic
joined_df = join_datasets(retention_df, bob_df)
print(f"Joined dataset shape: {joined_df.shape}")
print("Account-level churn distribution:")
print(joined_df[['account_number', 'churn_category']].drop_duplicates()['churn_category'].value_counts())

os.makedirs('../../data/02_intermediate', exist_ok=True)
joined_df.to_csv('../../data/02_intermediate/joined_data.csv', index=False)

print("Joined dataset saved to ../../data/02_intermediate/joined_data.csv")
print(f"Null counts after filling:\n{joined_df.isnull().sum()}")

Joined dataset shape: (715965, 40)
Account-level churn distribution:
churn_category
2    16233
0     4226
1     1067
Name: count, dtype: int64
Joined dataset saved to ../../data/02_intermediate/joined_data.csv
Null counts after filling:
case_title                       0
pull_van                         0
new_van                          0
van                              0
number_of_contracts              0
machines                         0
pull_type                        0
case_type                        0
risk                             0
current_status                   0
resolution_status                0
number_of_repair_cases           0
number_of_overdueservices        0
companysize                      0
customer_tier                    0
case_origin                      0
case_creation_date               0
resolved_date                    0
registered_date                  0
expected_pull_date               0
account_number                   0
company_sizing              

In [8]:
# Check partial churn accounts
partial_churn_agg = joined_df.groupby('account_number').agg({
    'lost_agreements': 'first',
    'active_agreements': 'first',
    'churn_category': 'first'
}).query('churn_category == 1')
print("Sample partial churn accounts:")
print(partial_churn_agg.head(10))
print(f"\nAre lost_agreements equal to active_agreements for all partial churn? { (partial_churn_agg['lost_agreements'] == partial_churn_agg['active_agreements']).all() }")

Sample partial churn accounts:
                   lost_agreements  active_agreements  churn_category
account_number                                                       
UK02-CGBA000139-L               80                 80               1
UK02-CGBA000143-L               99                 99               1
UK02-CGBA000270-L               45                 45               1
UK02-CGBA000323-L               27                 27               1
UK02-CGBA000336-L              518                518               1
UK02-CGBA000364-L               11                 11               1
UK02-CGBA000448-L                2                  2               1
UK02-CGBA000481-L                2                  2               1
UK02-CGBA000489-L                8                  8               1
UK02-CGBA000513-L                2                  2               1

Are lost_agreements equal to active_agreements for all partial churn? True
